## Baseline Classifier
Frozen sentence transformer embeddings (all-MiniLM-L6-v2) + logistic regression.
Embeddings are computed once and reused across all augmentation experiments.

In [1]:
import os

In [2]:
import pandas as pd
import numpy as np

In [3]:
os.chdir("..")

train_df = pd.read_parquet("data/processed/train.parquet")
test_df = pd.read_parquet("data/processed/test.parquet")
cal_df = pd.read_parquet("data/processed/calibration.parquet")

print(f"Train: {len(train_df)} | Positives: {train_df['label'].sum()}")
print(f"Calibration: {len(cal_df)} | Positives: {cal_df['label'].sum()}")
print(f"Test: {len(test_df)} | Positives: {test_df['label'].sum()}")

Train: 69510 | Positives: 674
Calibration: 9931 | Positives: 96
Test: 19861 | Positives: 192


## Compute Sentence Embeddings
Encode all three splits using a frozen sentence transformer (all-MiniLM-L6-v2).
Embeddings are computed once here and reused across all augmentation experiments,
ensuring fair comparison across conditions. No fine-tuning — the encoder is used
purely as a fixed feature extractor.

In [4]:
from sentence_transformers import SentenceTransformer

c:\Users\vkamat01\hedging-txtclf-experiments\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load frozen encoder — same model used for UMAP visualization
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode each split — show progress bar for transparency
print("Encoding train...")
X_train = model.encode(train_df['sentence'].tolist(), show_progress_bar=True)

print("Encoding calibration...")
X_cal = model.encode(cal_df['sentence'].tolist(), show_progress_bar=True)

print("Encoding test...")
X_test = model.encode(test_df['sentence'].tolist(), show_progress_bar=True)

y_train = train_df['label'].values
y_cal = cal_df['label'].values
y_test = test_df['label'].values

print(f"\nEmbedding shapes — Train: {X_train.shape} | Cal: {X_cal.shape} | Test: {X_test.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 513.10it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding train...


Batches: 100%|██████████| 2173/2173 [10:04<00:00,  3.60it/s]


Encoding calibration...


Batches: 100%|██████████| 311/311 [02:01<00:00,  2.57it/s]


Encoding test...


Batches: 100%|██████████| 621/621 [03:38<00:00,  2.84it/s]



Embedding shapes — Train: (69510, 384) | Cal: (9931, 384) | Test: (19861, 384)


## Save Embeddings to Disk
Embeddings are expensive to compute and identical across all experiments.
Saving them here ensures every augmentation condition uses the exact same
test and calibration representations, which is essential for fair comparison.
Train embeddings will be extended when synthetic samples are added later.